In [1]:
import pandas as pd
import numpy as np

In [2]:
 df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [11]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [12]:
# feature scaling
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state = 42
)

In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [16]:
# 7 different categories hence 7 layers in last layer

***
## ANN FOR CLASSIFICATION
***

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset

In [18]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype= torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype= torch.long)

In [20]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [21]:
train_loader = DataLoader(train_dataset , batch_size =32, shuffle = True )
test_loader = DataLoader(test_dataset , batch_size =32)


In [36]:
# building Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64,7)
        )

    def forward(self , x):
        return self.model(x)
        

In [37]:
model = ANN()


# loss & optim

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [38]:
# taining NN

epochs = 120

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()

        optimizer.step() # params update

        running_loss += loss.item() #convert to float

    train_loss = running_loss / len(train_loader) # loss of 1 epoch

    print(f"epoch={epoch+1}/{epochs}, loss = {train_loss}")

epoch=1/120, loss = 1.744189044703608
epoch=2/120, loss = 1.169084123943163
epoch=3/120, loss = 0.7851015627384186
epoch=4/120, loss = 0.5658419903205789
epoch=5/120, loss = 0.46067778312641644
epoch=6/120, loss = 0.3994118169597957
epoch=7/120, loss = 0.35954644692980725
epoch=8/120, loss = 0.33602369544298755
epoch=9/120, loss = 0.3038590641125389
epoch=10/120, loss = 0.2864755123205807
epoch=11/120, loss = 0.268564579927403
epoch=12/120, loss = 0.2500062499033368
epoch=13/120, loss = 0.2362494582067365
epoch=14/120, loss = 0.2303022849171058
epoch=15/120, loss = 0.22159875087116077
epoch=16/120, loss = 0.20429352525135744
epoch=17/120, loss = 0.20120258159611537
epoch=18/120, loss = 0.18768684339264166
epoch=19/120, loss = 0.18315642132707263
epoch=20/120, loss = 0.1797758496325949
epoch=21/120, loss = 0.17149665627790533
epoch=22/120, loss = 0.17113558413541835
epoch=23/120, loss = 0.16361523419618607
epoch=24/120, loss = 0.15442116574748702
epoch=25/120, loss = 0.1498230101297731


In [40]:
# evaluation
model.eval()

total=0
correct=0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2,0.3,-0.5,....] - 7 values

        _, predicted = torch.max(outputs , 1) # torch.max returns max_val , index and since we need index  we are only storing the index value in predicted var

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch
print("accuracy_score" , correct/total * 100)


accuracy_score 93.33333333333333


In [41]:
# can also use metrics of sklearn